Loading the cleaned data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/nassau_clean.csv")

features = ["Product Name", "Factory", "Region", "Ship Mode", "Distance KM", "Units", "Sales"]
target = "Lead Time Clean"

X = df[features]
y = df[target]

print(X.shape, y.shape)
X.head()

(10194, 7) (10194,)


,Product Name,Factory,Region,Ship Mode,Distance KM,Units,Sales
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,Interior,Standard Class,1563.508797,2,6.50
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Interior,Standard Class,1160.082944,2,7.50
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Interior,Standard Class,2188.317712,3,10.47
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,Interior,Standard Class,2188.317712,3,10.80
4,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,Atlantic,Standard Class,1008.219033,3,11.25


Split the data and prepare the inputs

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_cols = ["Product Name", "Factory", "Region", "Ship Mode"]
num_cols = ["Distance KM", "Units", "Sales"]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols),
])

print(X_train.shape, X_test.shape)

(8155, 7) (2039, 7)


Training the three models and comparing them

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([("prep", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })
    fitted[name] = pipe

pd.DataFrame(results).round(3)

,Model,MAE,RMSE,R2
0,Linear Regression,0.926,1.106,0.621
1,Random Forest,0.937,1.166,0.578
2,Gradient Boosting,0.919,1.103,0.623


In [4]:
import joblib

joblib.dump(fitted["Gradient Boosting"], "../models/gradient_boosting.pkl")
pd.DataFrame(results).round(3).to_csv("../reports/model_results.csv", index=False)
print("saved")

saved
